In [ ]:
google_drive_mountpoint = "/content/drive"
sourcecode_url = "https://github.com/stonebo/Research-Estimator-xMem.git"
default_branch = "feat/llm"

In [1]:
import os
colab_enable = True if "COLAB_RELEASE_TAG" in os.environ else False
if colab_enable:
    from google.colab import drive, userdata
    from urllib.parse import urlparse
    from pathlib import Path
    dir_name = "repo-xmem"
    repo_url = urlparse(sourcecode_url)
    # prepare repo
    if not Path(os.getcwd()).joinpath(dir_name).is_dir():
      repo_url = f"https://{userdata.get('G_USER')}:{userdata.get('G_PAT')}@{repo_url.hostname}{repo_url.path}"
      !apt install git
      !git clone {repo_url} {dir_name}
    # change workdir
    %cd {dir_name}
    !git fetch origin
    !git checkout {default_branch}
    # install dependencies
    if Path(os.getcwd()).joinpath("requirement.txt").is_file():
        !pip install -r requirement.txt
    else:
        raise FileNotFoundError(f"missing requirement.txt file")
    # mount google drive
    if not Path(google_drive_mountpoint).is_dir():
        drive.mount(google_drive_mountpoint)


In [2]:
import logging
import time
import torch
from exp.run import ExperimentRun, SummarySectionName
from exp.config import LargeTransformerExperiments
logging.basicConfig(level=logging.ERROR)


In [3]:
models = [
	"deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
	"Qwen/Qwen3-4B",
	"meta-llama/Llama-3.2-3B-Instruct",
	"JetBrains/Mellum-4b-base",
	"facebook/opt-125m"
]
model = models[-1]
opt = "AdamW"
bs = 1
gpu_id = 1
fp16 = True
in_docker = False

In [4]:
config = LargeTransformerExperiments()
config.debug = False
config.repeats = 1
config.gpu_id = gpu_id
config.fp16 = fp16
exp = ExperimentRun(config=config)


In [ ]:
if colab_enable:
	output_dir = Path().home().joinpath(config.run_id)
	target_dir = Path(google_drive_mountpoint).joinpath("100-ResearchData/2025-Middleware-xMem LLM/005-CoLab", config.run_id)
	if target_dir.is_dir() is False:
		target_dir.mkdir(parents=True, exist_ok=True)
	# create a softlink for output dir
	target_dir.symlink_to(output_dir, target_is_directory=True)

In [5]:
exp.add_task(
	model_name=model,
	batch_size=bs,
	optimizer=opt,
	gpu_id=gpu_id
)

In [6]:
exp.run_group_truth(in_docker=True)
time.sleep(2)
torch.cuda.empty_cache()
time.sleep(2)


100%|██████████| 1/1 [00:00<00:00, 59.64it/s]


=============== Start massively run for GPU train ======================


100%|██████████| 1/1 [01:55<00:00, 115.71s/it]
0it [00:00, ?it/s]
0it [00:00, ?it/s]


In [15]:
result = exp.run_estimation(estimators=[SummarySectionName.solution], in_docker=in_docker)

  0%|          | 0/1 [00:00<?, ?it/s]

facebook/opt-125m estimated by solution
Solution: CPU-based Running...
Preparing facebook/opt-125m with fp16: True and optimiser: AdamW
Loaded facebook/opt-125m in data type: torch.float16
Training on CPU Started
Initializing Training...
Training...
Using Mixed Precision (FP16) Training loop.
loading Model...
Forwarding...
Backwarding...
Optimizing...
Forwarding...
Backwarding...
Optimizing...
Forwarding...
Backwarding...
Optimizing...
Solution: Estimating memory usage...
Preparing facebook/opt-125m with fp16: True and optimiser: AdamW
Loaded facebook/opt-125m in data type: torch.float16
SummarySectionName.solution: Verification of Estimated Memory 2.04 GB...
Preparing facebook/opt-125m with fp16: True and optimiser: AdamW
Loaded facebook/opt-125m in data type: torch.float16
Using Mixed Precision (FP16) Training loop.
loading Model...
Forwarding...
Backwarding...
Optimizing...
Forwarding...
Backwarding...
Optimizing...
Forwarding...
Backwarding...
Optimizing...
Forwarding...
Backwardin

100%|██████████| 1/1 [01:11<00:00, 71.45s/it]


In [ ]:
result = exp.run_estimation(
	estimators=[
		SummarySectionName.DNNmem,
		SummarySectionName.LLmem,
		SummarySectionName.schedtune
	],
	in_docker=True
)
